# TUP Detection Engine — Historical Baseline Report (DeBERTa)

**Authors:** Vargas Pech · Fernández Castillo · Ruiz Peña · Rejón Quintal  
**Date:** June 2026  
**Purpose:** Documents the progressive improvement of TUP detection from the legacy stack to the DeBERTa-based classifier. Superseded by the Sentinel v2 evaluation in `tup_detection_guard_benchmark_report.ipynb`.

---

### Baseline progression (PINT proxy dataset — 4,254 prompts)

| Milestone | Engine | PINT balanced |
|-----------|--------|---------------|
| Legacy stack (regex + NER + LLM judge) | Full stack | **56.7%** |
| Rules-only refresh | L1 regex | **78.2%** |
| **DeBERTa candidate** *(historical)* | L1 + DeBERTa | **99.2%** |

The DeBERTa candidate catches every attack (100% recall) while passing 98.4% of benign traffic — a +42.5 pp gain over the legacy full stack.

> **Scope note:** Results use the public PINT proxy dataset (`pint-public-full.yaml`), not Lakera's proprietary leaderboard set.  
> The published Lakera Guard baseline (~95%) is shown as a reference line only.


## Methodology

| Item | Value |
|------|-------|
| Metric | **PINT balanced** = mean(attack recall, benign specificity) |
| Dataset | PINT proxy — 4,254 prompts (240 attacks, 4,014 benign) |
| Classifier | `ProtectAI/deberta-v3-base-prompt-injection-v2` |
| Threshold | `0.71` (calibrated on dev split) |
| Layer 1 | Regex rules `tup-rule-0001`–`0011` + text normalization |

**Pipeline:**

```
Prompt → normalize → TUP Layer 1 (regex) → DeBERTa classifier → (optional L3 judge)
```

All figures load from pre-computed JSON in `notebooks/data/` — no model re-run required.


In [ ]:
%pip install -q matplotlib pandas

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = REPO_ROOT / "notebooks" / "data"

PALETTE = {
    "mint": "#00F5A0",
    "critical": "#FF4D6D",
    "warning": "#FFB020",
    "violet": "#8B7CF6",
    "blue": "#4D8AFF",
    "dim": "#6B7570",
    "legacy": "#4A524E",
    "surface": "#111613",
    "bg": "#0A0F0D",
}

plt.rcParams.update({
    "figure.facecolor": PALETTE["bg"],
    "axes.facecolor": PALETTE["surface"],
    "axes.edgecolor": "#232B27",
    "axes.labelcolor": "#FFFFFF",
    "text.color": "#FFFFFF",
    "xtick.color": "#9AA39D",
    "ytick.color": "#9AA39D",
    "grid.color": "#232B27",
    "font.size": 11,
    "figure.dpi": 120,
})

def load_benchmark(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

phase4 = load_benchmark(DATA_DIR / "pint-benchmark-phase4-full.json")
comparison = load_benchmark(DATA_DIR / "pint-benchmark-comparison.json")
deepset = load_benchmark(DATA_DIR / "external" / "results" / "deepset-full.json")

print(f"Dataset: {phase4['rows']} rows | attacks={phase4['attacks']} benign={phase4['benign']}")
print(f"Classifier threshold: {phase4['injection_threshold']}")

In [ ]:
# Unified engine table: legacy baselines + current benchmark runs
LEGACY_ROWS = [
    {"engine": "Legacy L1 (regex, pre-rules)", "group": "legacy", "pint_balanced_pct": 62.6, "attack_detection_pct": 25.4, "benign_pass_pct": 99.8, "fn": None, "fp": None},
    {"engine": "Legacy L1 + NER", "group": "legacy", "pint_balanced_pct": 56.72, "attack_detection_pct": 27.1, "benign_pass_pct": 86.3, "fn": None, "fp": None},
    {"engine": "Legacy L3 (LLM judge)", "group": "legacy", "pint_balanced_pct": 50.62, "attack_detection_pct": 1.2, "benign_pass_pct": 100.0, "fn": None, "fp": None},
    {"engine": "Legacy full stack", "group": "legacy", "pint_balanced_pct": 56.72, "attack_detection_pct": 27.1, "benign_pass_pct": 86.3, "fn": None, "fp": None},
]

LABEL_MAP = {
    "TUP Layer 1 (regex)": "TUP L1 (rules v2)",
    "TUP Layer 1+Classifier": "TUP L1 + DeBERTa ★",
    "TUP Classifier only": "TUP DeBERTa only",
}

current_rows = []
for r in phase4["results"]:
    current_rows.append({
        "engine": LABEL_MAP.get(r["engine"], r["engine"]),
        "group": "current",
        "pint_balanced_pct": r["pint_balanced_pct"],
        "attack_detection_pct": r["attack_detection_pct"],
        "benign_pass_pct": r["benign_pass_pct"],
        "fn": r["fn"],
        "fp": r["fp"],
        "tp": r["tp"],
        "tn": r["tn"],
    })

df = pd.DataFrame(LEGACY_ROWS + current_rows)
winner = df.loc[df["pint_balanced_pct"].idxmax()]
legacy_best = df[df["group"] == "legacy"]["pint_balanced_pct"].max()
delta_vs_legacy = winner["pint_balanced_pct"] - legacy_best

display_cols = ["engine", "pint_balanced_pct", "attack_detection_pct", "benign_pass_pct", "fn", "fp"]
summary = df[display_cols].copy()
summary.columns = ["Engine", "PINT balanced %", "Attack recall %", "Benign pass %", "FN", "FP"]
summary.style.format({
    "PINT balanced %": "{:.2f}",
    "Attack recall %": "{:.1f}",
    "Benign pass %": "{:.1f}",
}, na_rep="—").background_gradient(subset=["PINT balanced %"], cmap="Greens")

## Headline metrics

The four numbers that matter for the production candidate, at a glance.

In [ ]:
w = df[df["engine"] == "TUP L1 + DeBERTa ★"].iloc[0]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
fig.suptitle("TUP L1 + DeBERTa — PINT proxy benchmark (4,254 rows)", fontsize=14, fontweight="bold", y=1.05)

kpis = [
    (f"{w['pint_balanced_pct']:.1f}%", "PINT balanced", PALETTE["mint"]),
    (f"{w['attack_detection_pct']:.0f}%", "Attack recall", PALETTE["blue"]),
    (f"{w['benign_pass_pct']:.1f}%", "Benign pass rate", PALETTE["violet"]),
    (f"+{delta_vs_legacy:.1f} pp", "vs legacy best", PALETTE["warning"]),
]

for ax, (value, label, color) in zip(axes, kpis):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.text(0.5, 0.58, value, ha="center", va="center", fontsize=26, fontweight="bold", color=color)
    ax.text(0.5, 0.22, label, ha="center", va="center", fontsize=11, color="#9AA39D")
    rect = mpatches.FancyBboxPatch((0.05, 0.05), 0.9, 0.9, boxstyle="round,pad=0.02", linewidth=1, edgecolor="#232B27", facecolor=PALETTE["surface"])
    ax.add_patch(rect)

plt.tight_layout()
plt.show()

## PINT balanced score across all engines

Every engine we tested, ranked. The dashed line marks the published Guard reference (~95% on the official PINT); the dotted line is our internal 90% target.

In [ ]:
plot_df = df.sort_values("pint_balanced_pct", ascending=True).reset_index(drop=True)
colors = [
    PALETTE["mint"] if "★" in e else (PALETTE["blue"] if g == "current" else PALETTE["legacy"])
    for e, g in zip(plot_df["engine"], plot_df["group"])
]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(plot_df["engine"], plot_df["pint_balanced_pct"], color=colors, height=0.65)

for bar, val in zip(bars, plot_df["pint_balanced_pct"]):
    ax.text(val + 0.4, bar.get_y() + bar.get_height() / 2, f"{val:.1f}%", va="center", fontsize=10, color="#FFFFFF")

ax.axvline(95.0, color=PALETTE["warning"], linestyle="--", linewidth=1.2, alpha=0.85)
ax.axvline(90.0, color=PALETTE["dim"], linestyle=":", linewidth=1, alpha=0.7)

ax.set_xlabel("PINT balanced score (%)")
ax.set_title("Detection engine comparison — open PINT proxy dataset", fontweight="bold", pad=12)
ax.set_xlim(45, 102)
ax.grid(axis="x", alpha=0.35)

legacy_patch = mpatches.Patch(color=PALETTE["legacy"], label="Legacy stack")
current_patch = mpatches.Patch(color=PALETTE["blue"], label="Current TUP (rules / ML)")
winner_patch = mpatches.Patch(color=PALETTE["mint"], label="Winner: L1 + DeBERTa")
ref_line = plt.Line2D([0], [0], color=PALETTE["warning"], linestyle="--", label="Published Guard ref. ~95% (official PINT)")
target_line = plt.Line2D([0], [0], color=PALETTE["dim"], linestyle=":", label="90% target")
ax.legend(handles=[legacy_patch, current_patch, winner_patch, ref_line, target_line], loc="lower right", framealpha=0.2)

plt.tight_layout()
plt.show()

## Attack recall vs benign pass rate

The balanced score hides a trade-off: a detector can win on recall by blocking aggressively, or win on pass rate by waving traffic through. Plotting the two axes separately shows where each engine actually lands. The winner is the point closest to the top-right corner — high recall and high pass rate at the same time.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 7))

for _, row in df.iterrows():
    is_winner = "★" in row["engine"]
    color = PALETTE["mint"] if is_winner else (PALETTE["blue"] if row["group"] == "current" else PALETTE["legacy"])
    size = 220 if is_winner else 120
    marker = "*" if is_winner else "o"
    ax.scatter(row["benign_pass_pct"], row["attack_detection_pct"], s=size, c=color, marker=marker, edgecolors="#FFFFFF", linewidths=0.6, zorder=3)
    offset = (4, 4) if not is_winner else (6, 6)
    short = row["engine"].replace(" ★", "")
    ax.annotate(short, (row["benign_pass_pct"], row["attack_detection_pct"]), textcoords="offset points", xytext=offset, fontsize=8.5, color="#C8CFCC")

ax.axhline(90, color=PALETTE["dim"], linestyle=":", alpha=0.6)
ax.axvline(90, color=PALETTE["dim"], linestyle=":", alpha=0.6)
ax.set_xlabel("Benign pass rate (%)")
ax.set_ylabel("Attack recall (%)")
ax.set_title("Recall vs specificity trade-off", fontweight="bold")
ax.set_xlim(82, 101)
ax.set_ylim(0, 105)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Improvement over the legacy stack

Three snapshots of the pipeline as it evolved: the old full stack, the rules refresh, and the current classifier-backed candidate. The left panel tracks the balanced score; the right tracks attack recall, where the jump is sharpest.

In [ ]:
milestones = pd.DataFrame([
    {"phase": "Legacy full stack", "pint_balanced_pct": 56.72, "attack_detection_pct": 27.1},
    {"phase": "Rules refresh (L1 v2)", "pint_balanced_pct": 78.22, "attack_detection_pct": 56.7},
    {"phase": "L1 + DeBERTa", "pint_balanced_pct": 99.18, "attack_detection_pct": 100.0},
])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

x = np.arange(len(milestones))
ax1.bar(x, milestones["pint_balanced_pct"], color=[PALETTE["legacy"], PALETTE["blue"], PALETTE["mint"]], width=0.55)
ax1.set_xticks(x)
ax1.set_xticklabels(milestones["phase"], rotation=15, ha="right")
ax1.set_ylabel("PINT balanced (%)")
ax1.set_title("Pipeline evolution", fontweight="bold")
ax1.set_ylim(0, 105)
for i, v in enumerate(milestones["pint_balanced_pct"]):
    ax1.text(i, v + 1.5, f"{v:.1f}%", ha="center", fontsize=10)
ax1.grid(axis="y", alpha=0.3)

ax2.bar(x, milestones["attack_detection_pct"], color=[PALETTE["legacy"], PALETTE["blue"], PALETTE["mint"]], width=0.55)
ax2.set_xticks(x)
ax2.set_xticklabels(milestones["phase"], rotation=15, ha="right")
ax2.set_ylabel("Attack recall (%)")
ax2.set_title("Attack detection rate", fontweight="bold")
ax2.set_ylim(0, 105)
for i, v in enumerate(milestones["attack_detection_pct"]):
    ax2.text(i, v + 1.5, f"{v:.1f}%", ha="center", fontsize=10)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Confusion matrix — TUP L1 + DeBERTa

Where the errors actually fall. With no false negatives, nothing slips past; the only cost is the handful of benign prompts in the false-positive cell, which we return to under Limitations.

In [ ]:
w = df[df["engine"] == "TUP L1 + DeBERTa ★"].iloc[0]
cm = np.array([[w["tn"], w["fp"]], [w["fn"], w["tp"]]])
labels = [["TN\n(benign OK)", "FP\n(benign blocked)"], ["FN\n(attack missed)", "TP\n(attack caught)"]]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Greens", vmin=0, vmax=cm.max())

for i in range(2):
    for j in range(2):
        val = int(cm[i, j])
        color = "#0A0F0D" if val > cm.max() * 0.5 else "#FFFFFF"
        ax.text(j, i, f"{labels[i][j]}\n{val:,}", ha="center", va="center", fontsize=12, color=color, fontweight="bold")

ax.set_xticks([])
ax.set_yticks([])
ax.set_title(f"Confusion matrix — {int(phase4['rows']):,} prompts", fontweight="bold")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print(f"False negatives: {int(w['fn'])} | False positives: {int(w['fp'])}")

## External sanity check — deepset/prompt-injections

A near-perfect score on one dataset can just mean the detector has overfit to it. To check that, we ran the same frozen config (`threshold=0.71`, no retuning) against an independent HuggingFace dataset of 662 rows. We expect lower numbers here, and that's the point — the gap tells us where the engine is still weak, which so far is multilingual false negatives and subtle roleplay attacks.

In [ ]:
deepset_hit = next(r for r in deepset["results"] if r["engine"] == "TUP Layer 1+Classifier")
pint_hit = next(r for r in phase4["results"] if r["engine"] == "TUP Layer 1+Classifier")

ext_df = pd.DataFrame([
    {"dataset": "PINT proxy (open)", "rows": phase4["rows"], "pint_balanced_pct": pint_hit["pint_balanced_pct"], "attack_detection_pct": pint_hit["attack_detection_pct"]},
    {"dataset": "deepset/prompt-injections", "rows": deepset["rows"], "pint_balanced_pct": deepset_hit["pint_balanced_pct"], "attack_detection_pct": deepset_hit["attack_detection_pct"]},
])

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(ext_df))
w_bar = 0.35
ax.bar(x - w_bar/2, ext_df["pint_balanced_pct"], w_bar, label="PINT balanced", color=PALETTE["mint"])
ax.bar(x + w_bar/2, ext_df["attack_detection_pct"], w_bar, label="Attack recall", color=PALETTE["blue"])
ax.set_xticks(x)
ax.set_xticklabels([f"{r['dataset']}\n(n={r['rows']})" for _, r in ext_df.iterrows()])
ax.set_ylabel("Score (%)")
ax.set_title("L1 + DeBERTa — in-distribution vs external dataset", fontweight="bold")
ax.legend(framealpha=0.2)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Limitations and next steps

A few things to keep in mind before reading these numbers as production guarantees:

- **Dataset scope.** Everything above is on the public PINT proxy (4,254 rows), not the proprietary Lakera leaderboard set, so the absolute scores are not directly comparable to the public leaderboard.
- **Evaluation mode.** We score prompts in isolation. Production traffic also carries the victim model's response, which gives a real deployment more signal than this benchmark sees.
- **False positives.** 66 benign prompts are flagged at threshold 0.71. That is an acceptable cost for high-security modes and moves with the threshold, so lower-friction modes can trade a little recall for fewer blocks.
- **Generalization.** The external benchmarks (deepset, Antijection) score lower. Multilingual and subtle attacks are the main open gap and the focus of the next iteration.
- **LLM judge (L3).** Kept only as an optional fallback. As a primary layer it adds false positives without a matching recall gain, so it stays off the fast path.

Source artifacts: `notebooks/data/pint-benchmark-phase4-full.json`, `pint-benchmark-comparison.json`, `external/results/deepset-full.json`.

To reproduce the full benchmark from scratch:

```bash
source .venv-pint/bin/activate
export HF_HOME=notebooks/data/hf-cache
python scripts/run_pint_benchmark.py --phase 2 \
  --dataset notebooks/data/pint-public-full.yaml \
  --injection-threshold 0.71 \
  --results-out notebooks/data/pint-benchmark-phase4-full.json
```